# The Commodity Edge: Counting Barrels from Orbit

**Alternative data for crude oil traders, built on Wherobots.**

Every Wednesday at 10:30 AM ET, the EIA releases the weekly U.S. petroleum status
report. By then, the market has already moved — because a handful of well-resourced
trading desks have been counting barrels at Cushing, Oklahoma all week, directly
from satellite imagery.

This notebook tells that story, end to end, using Wherobots:

1. **Where the oil is** — locate the 90M-barrel Cushing tank-farm complex, the
   NYMEX WTI physical delivery point.
2. **How satellites see it** — query Sentinel-2 imagery over the tank farms
   via Wherobots' STAC reader.
3. **How fill levels are extracted** — each floating-roof tank casts a shadow
   whose length is a direct function of the ullage (empty headspace). We show
   the geometry, then use a demo time series of per-tank fill estimates.
4. **The leading indicator** — satellite-derived inventory vs. the official
   EIA Wednesday release, shifted one week forward. The market-moving data
   is in the market before the release.
5. **The signal** — a simple strategy: long WTI when Cushing drew down last
   week, short when it built.
6. **The deliverable** — a GeoJSON feed of per-tank fill levels and derived
   signals, ready for the trading desk's web dashboard.

> **Demo scope.** The tank-farm AOIs are real public locations. The weekly
> fill estimates and WTI prices below are synthesized to illustrate the
> analytical workflow — the point is the *pipeline*, not the numbers. A
> production deployment would plug RasterFlow + SAM3 tank detection and a
> live WTI feed into the same Wherobots SQL surface.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. Where the Oil Is — Cushing Tank Farms

Cushing, Oklahoma holds roughly 90 million barrels of working storage across
eight major terminal operators. The table below pins each operator's tank
farm centroid and its approximate working capacity — public facility
disclosures. A ~5 km AOI around the hub captures all of them.

In [ ]:
CUSHING_HUB = (-96.767, 35.985)         # lon, lat
CUSHING_AOI_RADIUS_DEG = 0.05            # ~5 km

tank_farms = [
    # operator,                       lon,        lat,    tanks, capacity_mmbbl
    ("Enbridge Cushing Terminal",     -96.7500, 36.0050,    88,  20.1),
    ("Plains All American Cushing",   -96.7730, 35.9810,    72,  17.4),
    ("Enterprise Cushing",            -96.7580, 35.9680,    44,  11.8),
    ("Magellan East Cushing",         -96.7420, 35.9930,    36,   9.2),
    ("SemGroup White Cliffs",         -96.7890, 35.9900,    28,   7.5),
    ("BlueKnight Cushing South",      -96.7810, 35.9600,    24,   6.8),
    ("Phillips 66 Cushing",           -96.7380, 35.9770,    20,   5.4),
    ("Rose Rock Midstream",           -96.7610, 36.0110,    16,   4.1),
]

tank_schema = StructType([
    StructField("operator",        StringType()),
    StructField("lon",             DoubleType()),
    StructField("lat",             DoubleType()),
    StructField("tank_count",      IntegerType()),
    StructField("capacity_mmbbl",  DoubleType()),
])

tank_farms_df = sedona.createDataFrame(tank_farms, tank_schema) \
    .withColumn("geometry", f.expr("ST_Point(lon, lat)"))

tank_farms_df.createOrReplaceTempView("tank_farms")

totals = sedona.sql("""
    SELECT
        COUNT(*)             AS operators,
        SUM(tank_count)      AS total_tanks,
        ROUND(SUM(capacity_mmbbl), 1) AS working_capacity_mmbbl
    FROM tank_farms
""").toPandas().iloc[0]

print(f"Cushing hub: {totals.operators} operators, "
      f"{int(totals.total_tanks)} major tanks, "
      f"{totals.working_capacity_mmbbl} MMbbl working capacity")
tank_farms_df.select("operator", "tank_count", "capacity_mmbbl").show(truncate=False)

## 3. How Satellites See It — Sentinel-2 over Cushing

Sentinel-2 revisits Cushing every ~5 days at 10 m resolution. The cell below
hits the public AWS Element 84 STAC endpoint and counts how many cloud-free
scenes intersect our AOI over the past year — the raw supply of observations
a production pipeline would draw from.

In [ ]:
from sedona.stac.client import Client

cushing_bbox = [
    CUSHING_HUB[0] - CUSHING_AOI_RADIUS_DEG,
    CUSHING_HUB[1] - CUSHING_AOI_RADIUS_DEG,
    CUSHING_HUB[0] + CUSHING_AOI_RADIUS_DEG,
    CUSHING_HUB[1] + CUSHING_AOI_RADIUS_DEG,
]

stac = Client.open("https://earth-search.aws.element84.com/v1")

# Try a few datetime formats; the Sedona STAC client masks the underlying
# endpoint error so we fall back to an annual window known to work.
items_list = []
for dt_expr in [
    "2025-04-01T00:00:00Z/2026-04-01T00:00:00Z",
    "2025-04-01/2026-04-01",
    "2025",
]:
    try:
        items = stac.search(
            collection_id="sentinel-2-c1-l2a",
            bbox=cushing_bbox,
            datetime=dt_expr,
            max_items=200,
            return_dataframe=False,
        )
        items_list = list(items)
        print(f"STAC search succeeded with datetime={dt_expr!r}")
        break
    except Exception as e:
        print(f"  retry: {dt_expr!r} -> {type(e).__name__}")

if items_list:
    print(f"\nSentinel-2 scenes over Cushing: {len(items_list)}")
    print("\nMost recent scenes:")
    for item in sorted(items_list, key=lambda x: x.datetime, reverse=True)[:5]:
        print(f"  {item.datetime.strftime('%Y-%m-%d')}  {item.id}")
else:
    print("\nSTAC endpoint unavailable — continuing with demo narrative.")
    print("Sentinel-2 nominally revisits Cushing every ~5 days at 10 m,")
    print("yielding ~60-70 cloud-free observations per year.")

## 4. How Fill Levels Are Extracted — the Shadow Method

Crude oil floating-roof tanks have a roof that rides on the product surface.
As the tank drains, the roof drops, and the tank wall casts a longer inner
shadow. Given solar azimuth/elevation (known at every timestamp) and tank
geometry (diameter ≈ 80 m, height ≈ 20 m for typical 500k-bbl tanks), the
ullage — and thus the fill fraction — follows directly from the shadow
length. This is the method commercialized by Orbital Insight, Ursa Space,
and others.

The diagram below sketches the geometry.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (fill_frac, title) in zip(
    axes,
    [(0.85, "Tank 85% full — short shadow"),
     (0.25, "Tank 25% full — long shadow")]
):
    tank_h = 1.0
    roof_h = fill_frac * tank_h
    sun_elev_deg = 45
    shadow_len = (tank_h - roof_h) / np.tan(np.radians(sun_elev_deg))

    # Tank wall (cross-section)
    ax.add_patch(plt.Rectangle((0, 0), 1.5, tank_h,
                               fill=False, edgecolor='#333', linewidth=2))
    # Crude (below roof)
    ax.add_patch(plt.Rectangle((0, 0), 1.5, roof_h,
                               facecolor='#2a1a0a', alpha=0.85))
    # Floating roof
    ax.add_patch(plt.Rectangle((0, roof_h - 0.02), 1.5, 0.04,
                               facecolor='#888', edgecolor='#333'))
    # Shadow on the roof
    ax.add_patch(plt.Rectangle((0, roof_h), shadow_len * 1.5, tank_h - roof_h,
                               facecolor='#000', alpha=0.35))
    # Sun rays
    for x0 in np.linspace(-0.3, 0, 6):
        ax.plot([x0, x0 + (tank_h - roof_h)],
                [tank_h + 0.2, roof_h], color='#f7b500', alpha=0.6, lw=1)

    ax.set_xlim(-0.4, 2.0); ax.set_ylim(0, 1.4)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_visible(False)

plt.suptitle("Floating-roof tank: shadow length ∝ ullage ∝ (1 − fill)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("cushing_shadow_method.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Weekly Fill Estimates — Demo Time Series

In production, the pipeline would run RasterFlow + SAM3 to segment every
tank roof in every Sentinel-2 scene, measure shadow length, and solve for
fill. Here we skip the pixel math and use a 78-week synthesized series that
reproduces the seasonality and magnitude of EIA's Cushing stocks report.

In [ ]:
WEEKS = 78
rng = np.random.default_rng(42)
week_idx = np.arange(WEEKS)
weeks = pd.date_range("2024-10-18", periods=WEEKS, freq="W-FRI")

# Aggregate hub fill: seasonal + cyclical + noise, bounded [0.30, 0.85]
season   = 0.08 * np.sin(2 * np.pi * week_idx / 52 - 0.7)
cycle    = 0.06 * np.sin(2 * np.pi * week_idx / 14)
trend    = -0.0015 * week_idx
noise    = rng.normal(0, 0.015, WEEKS)
hub_fill = np.clip(0.62 + season + cycle + trend + noise, 0.30, 0.85)

# Per-tank-farm series: hub trajectory + operator-specific jitter
fill_rows = []
for operator, lon, lat, n_tanks, cap in tank_farms:
    op_noise  = rng.normal(0, 0.04, WEEKS)
    op_bias   = rng.uniform(-0.05, 0.05)
    op_series = np.clip(hub_fill + op_bias + op_noise, 0.15, 0.95)
    for w, date, fill in zip(week_idx, weeks, op_series):
        fill_rows.append((
            date.strftime("%Y-%m-%d"), operator, lon, lat, n_tanks, cap,
            float(fill), float(fill * cap)
        ))

fill_schema = StructType([
    StructField("week",           StringType()),
    StructField("operator",       StringType()),
    StructField("lon",            DoubleType()),
    StructField("lat",            DoubleType()),
    StructField("tank_count",     IntegerType()),
    StructField("capacity_mmbbl", DoubleType()),
    StructField("fill_fraction",  DoubleType()),
    StructField("stored_mmbbl",   DoubleType()),
])

fill_df = sedona.createDataFrame(fill_rows, fill_schema)
fill_df.createOrReplaceTempView("tank_fills")

print(f"{fill_df.count():,} weekly per-operator fill observations")
fill_df.filter(f.col("week") == weeks[-1].strftime("%Y-%m-%d")) \
       .select("operator", "fill_fraction", "stored_mmbbl") \
       .show(truncate=False)

## 6. The Leading Indicator — Satellite vs. EIA

Aggregate the per-operator fills into a hub total, convert to a WTI price
series, and compare against an EIA-style weekly release that lags by a
week. The satellite view sits in the market's hands days before the
official print.

In [ ]:
hub_weekly_df = sedona.sql("""
    SELECT
        week,
        ROUND(SUM(stored_mmbbl), 2) AS satellite_stocks_mmbbl,
        ROUND(SUM(stored_mmbbl) / SUM(capacity_mmbbl), 4) AS hub_fill_fraction
    FROM tank_fills
    GROUP BY week
    ORDER BY week
""").toPandas()
hub_weekly_df["week"] = pd.to_datetime(hub_weekly_df["week"])

# EIA release — same measurement, published one week later, small rounding
hub_weekly_df["eia_stocks_mmbbl"] = (
    hub_weekly_df["satellite_stocks_mmbbl"].shift(1)
    + rng.normal(0, 0.3, len(hub_weekly_df))
).round(2)

# WTI front-month price: anti-correlated with stocks (drawdowns → higher prices)
stocks_norm = (hub_weekly_df["satellite_stocks_mmbbl"]
               - hub_weekly_df["satellite_stocks_mmbbl"].mean()) \
              / hub_weekly_df["satellite_stocks_mmbbl"].std()
hub_weekly_df["wti_price"] = (
    78.0
    - 4.5 * stocks_norm.shift(1).fillna(0)       # prices react after 1 week
    + np.cumsum(rng.normal(0, 0.6, len(hub_weekly_df)))
).round(2)

fig, ax1 = plt.subplots(figsize=(12, 4.5))
ax1.plot(hub_weekly_df["week"], hub_weekly_df["satellite_stocks_mmbbl"],
         color="#1f77b4", lw=2.2, label="Satellite-derived stocks")
ax1.plot(hub_weekly_df["week"], hub_weekly_df["eia_stocks_mmbbl"],
         color="#888780", lw=1.6, ls="--", label="EIA weekly release (t+7)")
ax1.set_ylabel("Cushing stocks (MMbbl)", color="#1f77b4", fontsize=11)
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

ax2 = ax1.twinx()
ax2.plot(hub_weekly_df["week"], hub_weekly_df["wti_price"],
         color="#E24B4A", lw=1.8, alpha=0.9, label="WTI front-month ($/bbl)")
ax2.set_ylabel("WTI ($/bbl)", color="#E24B4A", fontsize=11)
ax2.tick_params(axis="y", labelcolor="#E24B4A")

ax1.set_title("Cushing storage: satellite vs. EIA release, with WTI price",
              fontsize=12, pad=10)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("cushing_satellite_vs_eia.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. The Signal — Satellite Δ-Stocks as a Trading Indicator

A naive rule: **if Cushing drew down last week (stocks ↓), go long WTI
next week; if it built (stocks ↑), go short.** Compare cumulative return
to buy-and-hold.

In [ ]:
bt = hub_weekly_df.copy()
bt["stocks_change"]     = bt["satellite_stocks_mmbbl"].diff()
bt["wti_ret"]           = bt["wti_price"].pct_change()
bt["signal"]            = -np.sign(bt["stocks_change"].shift(1))  # drawdown → long
bt["strategy_ret"]      = bt["signal"] * bt["wti_ret"]
bt["strategy_cum"]      = (1 + bt["strategy_ret"].fillna(0)).cumprod() - 1
bt["buyhold_cum"]       = (1 + bt["wti_ret"].fillna(0)).cumprod() - 1

hit_rate = (np.sign(bt["strategy_ret"]) == 1).sum() / bt["strategy_ret"].notna().sum()
sharpe   = (bt["strategy_ret"].mean() / bt["strategy_ret"].std()) * np.sqrt(52)
corr     = bt[["stocks_change", "wti_ret"]].shift(1).join(
              bt[["wti_ret"]].rename(columns={"wti_ret":"wti_fwd"})
           ).corr().loc["stocks_change","wti_fwd"]

print(f"Hit rate:                {hit_rate:.1%}")
print(f"Strategy Sharpe (ann.):  {sharpe:.2f}")
print(f"Corr(Δstocks_t-1, ret_t): {corr:+.2f}")
print(f"Cumulative strategy:     {bt['strategy_cum'].iloc[-1]:+.1%}")
print(f"Cumulative buy-and-hold: {bt['buyhold_cum'].iloc[-1]:+.1%}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(bt["week"], bt["strategy_cum"] * 100,
        color="#1f77b4", lw=2.2, label="Satellite Δ-stocks signal")
ax.plot(bt["week"], bt["buyhold_cum"] * 100,
        color="#888780", lw=1.6, ls="--", label="WTI buy-and-hold")
ax.axhline(0, color="#333", lw=0.6)
ax.set_title("Cumulative return: satellite-driven signal vs. passive WTI",
             fontsize=12, pad=8)
ax.set_ylabel("Cumulative return (%)")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("cushing_signal_backtest.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. The Deliverable — GeoJSON Feed for the Trading Desk

Package the latest per-operator fill level into a GeoJSON `FeatureCollection`
with the signal fields a frontend dashboard needs to color-code each tank
farm on a live map.

In [ ]:
latest_week = weeks[-1].strftime("%Y-%m-%d")
prior_week  = weeks[-2].strftime("%Y-%m-%d")

delta_sql = f"""
    SELECT
        c.operator,
        c.lon, c.lat,
        c.tank_count,
        c.capacity_mmbbl,
        c.fill_fraction        AS fill_current,
        p.fill_fraction        AS fill_prior,
        c.stored_mmbbl         AS stocks_current_mmbbl,
        ROUND(c.stored_mmbbl - p.stored_mmbbl, 3) AS stocks_delta_mmbbl,
        CASE
            WHEN c.stored_mmbbl - p.stored_mmbbl < -0.2 THEN 'BULLISH'
            WHEN c.stored_mmbbl - p.stored_mmbbl >  0.2 THEN 'BEARISH'
            ELSE                                             'NEUTRAL'
        END AS signal,
        ST_AsGeoJSON(ST_Point(c.lon, c.lat)) AS geojson
    FROM tank_fills c
    JOIN tank_fills p
      ON c.operator = p.operator
    WHERE c.week = '{latest_week}'
      AND p.week = '{prior_week}'
"""

latest_df = sedona.sql(delta_sql)
latest_df.select("operator", "fill_current",
                 "stocks_delta_mmbbl", "signal").show(truncate=False)

features = [
    {
        "type": "Feature",
        "properties": {
            "operator":             row.operator,
            "tank_count":           row.tank_count,
            "capacity_mmbbl":       row.capacity_mmbbl,
            "fill_current":         round(row.fill_current, 4),
            "fill_prior":           round(row.fill_prior, 4),
            "stocks_current_mmbbl": round(row.stocks_current_mmbbl, 3),
            "stocks_delta_mmbbl":   row.stocks_delta_mmbbl,
            "signal":               row.signal,
            "as_of":                latest_week,
        },
        "geometry": json.loads(row.geojson),
    }
    for row in latest_df.collect()
]

feature_collection = {"type": "FeatureCollection", "features": features}

out_path = "/tmp/cushing_tank_signal.geojson"
with open(out_path, "w") as fh:
    json.dump(feature_collection, fh, indent=2)

print(f"\nWrote {len(features)} tank-farm features to {out_path}")
print(json.dumps(feature_collection, indent=2)[:1500], "...")

## 9. Preview on a Map

In [ ]:
SedonaKepler.create_map(
    latest_df.select("operator", "fill_current", "stocks_delta_mmbbl",
                     "signal", "capacity_mmbbl",
                     f.expr("ST_Point(lon, lat)").alias("geometry")),
    name="Cushing Tank Farms — Latest Signal"
)

---

## From demo to production

Everything above is one Wherobots notebook — same surface scales to a live
commodity-data product:

| Demo step | Production replacement |
|---|---|
| Hardcoded operator centroids | RasterFlow + SAM3 tank segmentation on Sentinel-2 |
| Synthesized fill time series | Shadow-length inversion per tank, per scene |
| EIA series = satellite shifted | Live EIA Weekly Petroleum Status pull |
| Synthesized WTI prices | CME/ICE front-month feed |
| Static GeoJSON file | Scheduled job → S3 → frontend map |

## Outputs

| File | Contents |
|---|---|
| `cushing_shadow_method.png` | Explainer: shadow length ↔ tank fill |
| `cushing_satellite_vs_eia.png` | Satellite stocks vs. lagged EIA + WTI price |
| `cushing_signal_backtest.png` | Cumulative strategy vs. buy-and-hold |
| `/tmp/cushing_tank_signal.geojson` | Per-operator latest fill & trading signal |